In [ ]:
import numpy as np
import pandas as pd

# Define column names based on NASA specification
index_names = ["unit_nr", "time_cycles"]
setting_names = ["setting_1", "setting_2", "setting_3"]
sensor_names = [f"s_{i}" for i in range(1, 22)]
col_names = index_names + setting_names + sensor_names

# 1. Load Raw Training Data
train_path = "***/data_analysis_portfolio/project-2/data/raw/train_FD001.txt"
df_raw = pd.read_csv(train_path, sep=r"\s+", header=None, names=col_names)

# 2. Calculate Remaining Useful Life (RUL)
# For each engine, max(time_cycles) represents total life at failure.
# RUL_t = Max_Cycle_i - Current_Cycle_t
max_cycles = df_raw.groupby("unit_nr")["time_cycles"].max().reset_index()
max_cycles.rename(columns={"time_cycles": "max_life"}, inplace=True)

df_train = pd.merge(df_raw, max_cycles, on="unit_nr")
df_train["RUL"] = df_train["max_life"] - df_train["time_cycles"]
df_train.drop(columns=["max_life"], inplace=True)

# 3. Identify & Remove Zero-Variance Sensors (Flatlines)
# Sensors with standard deviation ~ 0 contain no physical degradation signal.
std_series = df_train[sensor_names].std()
flat_sensors = std_series[std_series < 0.01].index.tolist()

print(f"Flatline sensors detected (Zero Variance): {flat_sensors}")
df_cleaned = df_train.drop(columns=flat_sensors)

# 4. Save Processed Dataset & Feature Metadata
df_cleaned.to_csv("***/data_analysis_portfolio/project-2/data/processed/cmapss_cleaned.csv", index=False)
print(f"Cleaned dataset saved successfully! Shape: {df_cleaned.shape}")

Flatline sensors detected (Zero Variance): ['s_1', 's_5', 's_6', 's_10', 's_16', 's_18', 's_19']
Cleaned dataset saved successfully! Shape: (20631, 20)
